# Percie fine-tune 2/5 — Coder

Fine-tunes **Qwen2.5-Coder-3B-Instruct** on Python problems, with the app's exact system prompt and the practice pack's prompt style (`Complete this Python function:` + a stub with examples).

**Data:** MBPP `full` train+validation+prompt (474 problems, turned into stub prompts) + `Vezora/Tested-143k-Python-Alpaca` (tested Python solutions). **HumanEval is filtered out** of the training data. **Scored on:** MBPP `test` with its unit tests.

The stock model already scores 17/20 on the practice items, so the bar is high. Keep the fine-tune only if the app's eval improves.

**Run:** `Runtime → Change runtime type → A100 GPU` (T4 works, slower), then `Runtime → Run all`. Only the ⚙️ Settings cell is meant to be edited.

⚠️ **Hackathon rule:** train only on TRAIN splits. Every practice item comes from a validation/test split, and the judges' items almost certainly do too. Never train on the practice pack.

## 1. Install

In [ ]:
# Unsloth installs matching torch / transformers / trl / peft / bitsandbytes.
# If a new release breaks, pin the previous one, e.g. unsloth==2026.11.6
!pip install -q unsloth

## 2. ⚙️ Settings

In [ ]:
MODEL_NAME  = "unsloth/Qwen2.5-Coder-3B-Instruct"
RUN_NAME    = "neuro-coder"
OLLAMA_NAME = "neuro-coder"
APP_SETTING = "NEUROASSIST_CODE_MODEL"

SYSTEM_PROMPT = 'You are an expert Python programmer. Reply with ONE complete Python code block that contains the full function, including its signature and any imports it needs. No explanation, no tests, no example usage.'   # copied from the app (neuroassist/tools/code.py)

VEZORA_ROWS = 6000          # tested instruction → solution pairs
EVAL_ROWS   = 150           # MBPP test problems used for pass@1

MAX_SEQ_LENGTH = 2048
LORA_RANK, LORA_ALPHA = 16, 16
EPOCHS, MAX_STEPS = 1, None
LEARNING_RATE = 1e-4        # lower than usual: protect what the model already does well
BATCH_SIZE, GRAD_ACCUM = 8, 2
SEED = 3407
SAVE_TO_DRIVE, DRIVE_DIR = True, "/content/drive/MyDrive/neuroassist-finetunes"
GGUF_QUANT = "q4_k_m"

## 3. GPU and Google Drive

In [ ]:
import torch, os, json, random, re
assert torch.cuda.is_available(), "No GPU. Runtime → Change runtime type → GPU"
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name}, {gpu.total_memory / 2**30:.1f} GB")
if gpu.total_memory < 20 * 2**30 and BATCH_SIZE > 2:      # T4 / L4: keep the effective batch size
    GRAD_ACCUM, BATCH_SIZE = GRAD_ACCUM * BATCH_SIZE // 2, 2
    print(f"Small GPU: BATCH_SIZE=2, GRAD_ACCUM={GRAD_ACCUM}")
random.seed(SEED)

OUT_DIR = f"/content/{RUN_NAME}"
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")      # mounted first, so nothing is lost if a later cell fails
    OUT_DIR = f"{DRIVE_DIR}/{RUN_NAME}"
os.makedirs(OUT_DIR, exist_ok=True)
print("Saving to", OUT_DIR)

## 4. Load the model and add LoRA adapters

In [ ]:
from unsloth import FastLanguageModel, is_bf16_supported

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH, dtype=None, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=SEED)
model.print_trainable_parameters()


def chat(user, assistant=None, system=None):
    """One training/eval conversation in the app's format."""
    msgs = [{"role": "system", "content": system if system is not None else SYSTEM_PROMPT},
            {"role": "user", "content": user}]
    if assistant is not None:
        msgs.append({"role": "assistant", "content": assistant})
    return msgs


def generate_batch(conversations, max_new_tokens=256, batch_size=16):
    """Greedy answers for a list of conversations (without the assistant turn)."""
    FastLanguageModel.for_inference(model)
    tokenizer.padding_side = "left"
    outs = []
    for i in range(0, len(conversations), batch_size):
        prompts = [tokenizer.apply_chat_template(c, add_generation_prompt=True, tokenize=False)
                   for c in conversations[i:i + batch_size]]
        enc = tokenizer(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id)
        outs += tokenizer.batch_decode(out[:, enc.input_ids.shape[1]:], skip_special_tokens=True)
    tokenizer.padding_side = "right"
    FastLanguageModel.for_training(model)
    return [o.strip() for o in outs]

## 5. Build the training data and the held-out test

In [ ]:
import ast, subprocess, sys, tempfile
from datasets import load_dataset

# ---- HumanEval decontamination (HumanEval is used ONLY to filter it out) ----
def _toks(s):
    return re.findall(r"\w+|[^\w\s]", s)
humaneval = load_dataset("openai/openai_humaneval", split="test")
HE_NAMES = {r["entry_point"] for r in humaneval if len(r["entry_point"]) >= 8}
HE_GRAMS = set()
for r in humaneval:
    t = _toks(r["prompt"] + r["canonical_solution"])
    HE_GRAMS |= {" ".join(t[i:i + 12]) for i in range(len(t) - 11)}

def contaminated(text):
    if any(re.search(rf"\b{re.escape(n)}\b", text) for n in HE_NAMES):
        return True
    t = _toks(text)
    return any(" ".join(t[i:i + 12]) in HE_GRAMS for i in range(len(t) - 11))

def fence(code):
    return f"```python\n{code.strip()}\n```"

def mbpp_prompt(row):
    """MBPP problem → the practice pack's 'Complete this Python function' format."""
    code = row["code"].replace("\r\n", "\n").replace("\t", "    ")
    m = re.search(r"assert\s+(?:not\s+)?\(?\s*(\w+)\s*\(", row["test_list"][0])
    sig = m and re.search(rf"^def {m.group(1)}\s*\(.*?\)\s*(?:->.*?)?:", code, re.M | re.S)
    if not sig:
        return None, code
    shown = []
    for t in row["test_list"][:2]:
        mm = re.match(r"assert\s+(.*?)\s*==\s*(.*)$", t.strip())
        if mm:
            shown.append(f"    >>> {mm.group(1)}\n    {mm.group(2)}")
    imports = "\n".join(l for l in code.splitlines() if l.startswith(("import ", "from ")))
    stub = (imports + "\n\n\n" if imports else "") + sig.group(0) + '\n    """ ' + row["text"].strip() + "\n" \
           + "\n".join(shown) + '\n    """\n'
    return "Complete this Python function:\n\n\n" + stub, code

examples, dropped = [], 0
for split_name in ("train", "validation", "prompt"):
    for row in load_dataset("google-research-datasets/mbpp", "full", split=split_name):
        prompt, code = mbpp_prompt(row)
        if not prompt:
            continue
        if contaminated(prompt + code):
            dropped += 1
            continue
        examples.append(chat(prompt, fence(code)))
n_mbpp = len(examples)

def extract(out):
    blocks = re.findall(r"```(?:python|py)?\s*\n(.*?)```", out, re.S)
    code = max(blocks, key=len) if blocks else out
    try:
        ast.parse(code)
    except SyntaxError:
        return None
    return code if "def " in code and len(code) < 4000 else None

vez = load_dataset("Vezora/Tested-143k-Python-Alpaca", split="train", streaming=True).shuffle(seed=SEED, buffer_size=20000)
for row in vez:
    if len(examples) - n_mbpp >= VEZORA_ROWS:
        break
    instr = (row["instruction"] + ("\n\n" + row["input"] if row["input"] else "")).strip()
    code = extract(row["output"])
    if not code:
        continue
    if contaminated(instr + code):
        dropped += 1
        continue
    examples.append(chat(instr, fence(code)))
random.shuffle(examples)
print(f"MBPP {n_mbpp} · Vezora {len(examples) - n_mbpp} · dropped as HumanEval-like: {dropped}")

# ---- Held-out test: MBPP test split with its unit tests ----
mbpp_test = [r for r in load_dataset("google-research-datasets/mbpp", "full", split="test")
             if mbpp_prompt(r)[0]][:EVAL_ROWS]

def passes(code, row):
    src = (row["test_setup_code"] or "") + "\n" + code + "\n\n" + "\n".join(row["test_list"])
    with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False) as f:
        f.write(src)
    try:
        return subprocess.run([sys.executable, f.name], capture_output=True, timeout=10).returncode == 0
    except subprocess.TimeoutExpired:
        return False

def evaluate_model():
    outs = generate_batch([chat(mbpp_prompt(r)[0]) for r in mbpp_test], max_new_tokens=512, batch_size=16)
    ok = sum(passes(extract(o) or "", r) for o, r in zip(outs, mbpp_test))
    return {"MBPP test pass@1": round(ok / len(mbpp_test), 3), "passed": f"{ok}/{len(mbpp_test)}"}

## 6. Format with the model's chat template

In [ ]:
from datasets import Dataset

texts = [tokenizer.apply_chat_template(m, tokenize=False) for m in examples]
lengths = [len(tokenizer(t, add_special_tokens=False).input_ids) for t in texts]
keep = [t for t, l in zip(texts, lengths) if l <= MAX_SEQ_LENGTH]
print(f"{len(keep)} examples ({len(texts) - len(keep)} dropped as longer than {MAX_SEQ_LENGTH} tokens), "
      f"median {sorted(lengths)[len(lengths) // 2]} tokens")
data = Dataset.from_dict({"text": keep}).shuffle(seed=SEED)
split = data.train_test_split(test_size=min(0.02, 500 / len(data)), seed=SEED)
train_ds, val_ds = split["train"], split["test"]
print(f"train {len(train_ds)} · validation-loss set {len(val_ds)}")
print("\n--- one training example ---\n" + train_ds[0]["text"][:1500])

## 7. Score the base model (before)

In [ ]:
print("Scoring the base model before training (this is the number to beat)...")
before = evaluate_model()
print(json.dumps(before, indent=2))

## 8. Train

In [ ]:
import inspect
from trl import SFTTrainer, SFTConfig

wanted = dict(
    output_dir=f"/content/checkpoints/{RUN_NAME}",
    dataset_text_field="text", max_length=MAX_SEQ_LENGTH, max_seq_length=MAX_SEQ_LENGTH,
    packing=False, dataset_num_proc=2,
    per_device_train_batch_size=BATCH_SIZE, gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS, max_steps=MAX_STEPS if MAX_STEPS else -1,
    learning_rate=LEARNING_RATE, warmup_ratio=0.05, lr_scheduler_type="cosine",
    optim="adamw_8bit", weight_decay=0.01,
    bf16=is_bf16_supported(), fp16=not is_bf16_supported(),
    logging_steps=10, eval_strategy="steps", eval_steps=100, save_strategy="no",
    seed=SEED, report_to="none")
accepted = inspect.signature(SFTConfig.__init__).parameters      # TRL renames args between versions
if "max_length" in accepted:
    wanted.pop("max_seq_length")
args = SFTConfig(**{k: v for k, v in wanted.items() if k in accepted})

kw = dict(model=model, train_dataset=train_ds, eval_dataset=val_ds, args=args)
kw["processing_class" if "processing_class" in inspect.signature(SFTTrainer.__init__).parameters
   else "tokenizer"] = tokenizer
trainer = SFTTrainer(**kw)

# Loss only on the assistant's answer. Turn markers are read from this model's chat template.
from unsloth.chat_templates import train_on_responses_only
probe = tokenizer.apply_chat_template(
    [{"role": "user", "content": "@@U@@"}, {"role": "assistant", "content": "@@A@@"}], tokenize=False)
def _marker(text, words):
    idx = max(text.rfind(w) for w in words)
    return text[text.rfind("<", 0, idx):]
user_part = _marker(probe[:probe.index("@@U@@")], ["user"])
asst_part = _marker(probe[probe.index("@@U@@") + 5:probe.index("@@A@@")], ["assistant", "model"])
print(f"training on answers only: {user_part!r} … {asst_part!r}")
trainer = train_on_responses_only(trainer, instruction_part=user_part, response_part=asst_part)

stats = trainer.train()
print(f"done in {stats.metrics['train_runtime'] / 60:.1f} min · train loss {stats.metrics['train_loss']:.3f}")

## 9. Score again (after) and compare

In [ ]:
after = evaluate_model()
print(f"{'metric':<28}{'before':>10}{'after':>10}")
for k in before:
    b, a = before[k], after[k]
    print(f"{k:<28}{b:>10.3f}{a:>10.3f}" if isinstance(b, float) else f"{k:<28}{b!s:>10}{a!s:>10}")
json.dump({"model": MODEL_NAME, "before": before, "after": after}, open(f"{OUT_DIR}/scores.json", "w"), indent=2)
print(f"\nKeep this fine-tune only if it also beats the stock model in the app's eval (eval/run_eval.py).")

## 10. Save adapters, export GGUF and a Modelfile for Ollama

In [ ]:
import glob, shutil

model.save_pretrained(f"{OUT_DIR}/lora")
tokenizer.save_pretrained(f"{OUT_DIR}/lora")
print("LoRA adapters saved:", f"{OUT_DIR}/lora")

gguf_dir = f"/content/{RUN_NAME}_gguf"
model.save_pretrained_gguf(gguf_dir, tokenizer, quantization_method=GGUF_QUANT)
found = [f for f in glob.glob(f"/content/{RUN_NAME}_gguf*/**/*", recursive=True)
         if f.endswith(".gguf") or os.path.basename(f) == "Modelfile"]
for f in found:
    shutil.copy(f, OUT_DIR)
    print("copied", os.path.basename(f), f"({os.path.getsize(f) / 2**20:.0f} MB)")
ggufs = sorted(os.path.basename(f) for f in found if f.endswith(".gguf"))
assert ggufs, "No .gguf was produced; check the export log above."
gguf = next((g for g in ggufs if GGUF_QUANT.lower() in g.lower()), ggufs[0])

# Our own Modelfile, pointing at the quantized file. Qwen models use the ChatML format.
assert "qwen" in MODEL_NAME.lower(), "Modelfile below is for Qwen; write one for this model family."
modelfile = f"""FROM ./{gguf}
TEMPLATE \"\"\"{{{{- if .System }}}}<|im_start|>system
{{{{ .System }}}}<|im_end|>
{{{{ end }}}}{{{{- range .Messages }}}}<|im_start|>{{{{ .Role }}}}
{{{{ .Content }}}}<|im_end|>
{{{{ end }}}}<|im_start|>assistant
\"\"\"
PARAMETER stop "<|im_end|>"
PARAMETER stop "<|im_start|>"
PARAMETER temperature 0
"""
open(f"{OUT_DIR}/Modelfile", "w").write(modelfile)
print(f"""
Done. On the demo laptop, copy {gguf} and Modelfile from Drive into one folder, then:
    ollama create {OLLAMA_NAME} -f Modelfile
and run the app / eval with:
    $env:{APP_SETTING}="{OLLAMA_NAME}"
""")